# Create tables with DLC results, immobility removes (Cheeseboard : Opto et Opto_Chemo)

In [3]:
##### LOCAL #####

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
import math
from pathlib import Path
import re
from scipy.ndimage import median_filter
from scipy.interpolate import interp1d
from scipy.spatial import distance
import openpyxl
import warnings
warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)  # évite le spam 'DataFrame is highly fragmented'

##############################################################################
# Choose the folder containing the .h5 files
##############################################################################

folder_path = "//10.69.168.1/crnldata/forgetting/Carla/Cheeseboard/"
counter = 0
counterProbe = 0
day = 1
trial = 1
previousmice = 0
previous_session_time = 0
previous_session_type = None
previous_experiment = None
n_ignored_other_layout = 0   # .h5 hors Cheeseboard/Opto ou Cheeseboard/Opto_Chemo

# Dossier d'expérience  ->  nom utilisé dans les tableaux de sortie
experiment_tables = {
    "Opto":       "Opto",        # -> Summary_Table_Opto
    "Opto_Chemo": "Chemo_Opto",  # -> Summary_Table_Chemo_Opto
}
Summary_table = pd.DataFrame()
Sholl_table = pd.DataFrame()
skipped_files = []   # liste des vidéos NON analysées (avec la raison)

##############################################################################
# Immobilité (méthode du notebook DLC_5_create_table_final_filtered)
#   Une souris est immobile si elle reste dans un rayon de 'immobility_radius_cm' pendant au moins
#   'immobility_min_s' secondes consécutives. Ces frames sont retirées de la trajectoire.
#   Ce notebook produit UNIQUEMENT les tableaux avec l'immobilité retirée (suffixe '_immobility' à la fin des noms).
##############################################################################
immobility_radius_cm = 2     # rayon (5 cm de diamètre)
immobility_min_s     = 2       # durée minimale (secondes)

##############################################################################
# Informations par souris (implant, sexe) : fichier _MouseID_ImplantColorCode.csv
# colonnes attendues : Implant;MouseID;Genotype;Sex;Expe;Virus   (séparateur ';')
##############################################################################
mouse_info_path = "C:/Users/AudreyHay/Documents/Carla/Visual Code/HayLabAnalysis/python/_MouseID_ImplantColorCode.csv"

mouse_info = {}
if os.path.exists(mouse_info_path):
    try:
        info_df = pd.read_csv(mouse_info_path, sep=';', dtype=str, keep_default_na=False, encoding='utf-8-sig')
    except UnicodeDecodeError:
        info_df = pd.read_csv(mouse_info_path, sep=';', dtype=str, keep_default_na=False, encoding='latin-1')
    info_df = info_df.apply(lambda col: col.str.strip())                        # enlève les espaces ('WT ' -> 'WT')
    info_df = info_df.replace({'': np.nan, 'Nan': np.nan, 'nan': np.nan, 'NaN': np.nan})   # 'Nan' = pas d'implant
    mouse_info = info_df.set_index('MouseID').to_dict('index')
    print(f'Fichier souris chargé : {len(mouse_info)} souris ({mouse_info_path})')
else:
    print(f'/!\\ Fichier souris introuvable : {mouse_info_path}  -> colonnes implant / sex vides')

mice_not_in_info = set()     # souris présentes dans les vidéos mais absentes du fichier
info_mismatches  = []        # incohérences dossier vs fichier (génotype / virus)

##############################################################################
# Define functions
##############################################################################

def calculate_relative_distance(x1, y1, x2, y2):
    return math.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)

def calculate_distance_run(x_coords, y_coords):
    distances = np.sqrt(np.diff(x_coords) ** 2 + np.diff(y_coords) ** 2)
    distances_raw = distances.copy()   # voisins lus dans la version NON remplie : un trou long (ex. immobilité retirée) ne se propage pas
    for i in range(1, len(distances) - 1):
        if np.isnan(distances[i]):
            neighbors = [distances_raw[i - 1], distances_raw[i + 1]]
            valid_neighbors = [x for x in neighbors if not np.isnan(x)]
            if len(valid_neighbors) > 0:
                distances[i] = np.mean(valid_neighbors)
    total_distance_cm = np.nansum(distances) / pixel_to_cm
    return total_distance_cm, distances

def find_long_non_nan_sequences(arr, min_length=100):
    mask = ~np.isnan(arr)
    diff = np.diff(np.concatenate(([0], mask.astype(int), [0])))
    starts = np.where(diff == 1)[0]
    ends = np.where(diff == -1)[0]
    sequences = [arr[start:end] for start, end in zip(starts, ends) if (end - start) > min_length]
    return sequences

def remove_outliers_median_filter(data, window=1):
    data = np.array(data, dtype=float)
    filtered_data = np.copy(data)
    half_window = window // 2
    for i in range(len(data)):
        start = max(0, i - half_window)
        end = min(len(data), i + half_window + 1)
        local_values = data[start:end]
        if np.all(np.isnan(local_values)):
            median_value = np.nan
        else:
            median_value = np.nanmedian(local_values)
        if not np.isnan(data[i]):
            filtered_data[i] = median_value
    return filtered_data

def replace_high_speed_points_with_nan(x, y, speed_threshold):
    x = np.array(x, dtype='float')
    y = np.array(y, dtype='float')
    dx = np.diff(x)
    dy = np.diff(y)
    speeds = np.sqrt(dx**2 + dy**2)
    high_speed_mask = speeds > speed_threshold
    x_out = x.copy()
    y_out = y.copy()
    for i in range(len(high_speed_mask)):
        if high_speed_mask[i]:
            if i > 0 and i < len(x) - 1:
                if speeds[i] > speeds[i - 1]:
                    x_out[i + 1] = np.nan
                    y_out[i + 1] = np.nan
                else:
                    x_out[i] = np.nan
                    y_out[i] = np.nan
    return x_out, y_out

def interpolate_2d_path(x, y, kind='linear', fill='extrapolate'):
    x = np.array(x, dtype='float')
    y = np.array(y, dtype='float')
    indices = np.arange(len(x))
    valid_mask = ~np.isnan(x) & ~np.isnan(y)
    if np.sum(valid_mask) < 2:
        raise ValueError("Not enough valid points to interpolate/extrapolate.")
    interp_x = interp1d(indices[valid_mask], x[valid_mask], kind=kind, fill_value=fill, bounds_error=False)
    interp_y = interp1d(indices[valid_mask], y[valid_mask], kind=kind, fill_value=fill, bounds_error=False)
    x_filled = x.copy()
    y_filled = y.copy()
    nan_mask = np.isnan(x) | np.isnan(y)
    x_filled[nan_mask] = interp_x(indices[nan_mask])
    y_filled[nan_mask] = interp_y(indices[nan_mask])
    return x_filled, y_filled

def limit_speed(x, y, max_speed):
    dx = np.diff(x.copy())
    dy = np.diff(y.copy())
    speeds = np.sqrt(dx**2 + dy**2)
    for i, t in enumerate(speeds):
        if t > max_speed:
            x[i + 1] = x[i]
            y[i + 1] = y[i]
            if i + 2 < len(x):
                x[i + 2] = x[i]
                y[i + 2] = y[i]
    return x, y

def remove_short_sequences(arr, max_len=10):
    arr = np.array(arr, dtype='float')
    result = arr.copy()
    is_value = ~np.isnan(arr)
    i = 0
    while i < len(arr):
        if is_value[i]:
            start = i
            while i < len(arr) and is_value[i]:
                i += 1
            end = i
            seq_len = end - start
            if seq_len <= max_len:
                left_nan = (start == 0) or np.isnan(arr[start - 1])
                right_nan = (end == len(arr)) or np.isnan(arr[end])
                if left_nan and right_nan:
                    result[start:end] = np.nan
        else:
            i += 1
    return result

def apply_circular_mask(x, y, x_mask, y_mask, mask_radius):
    path_coords = np.column_stack((x, y))
    mask_center = np.array([x_mask, y_mask])
    dist_to_mask_center = distance.cdist(path_coords, [mask_center]).flatten()
    mask = dist_to_mask_center <= mask_radius
    return x[mask], y[mask]

def get_quadrant(x, y, table_center_x, table_center_y):
    dx = x - table_center_x
    dy = y - table_center_y
    diag1 = dx - dy
    diag2 = dx + dy
    if diag1 > 0 and diag2 > 0:
        return "South"
    elif diag1 < 0 and diag2 > 0:
        return "West"
    elif diag1 < 0 and diag2 < 0:
        return "North"
    else:
        return "East"

def compute_performance_metrics(
    individual_x, individual_y,
    start_frame, t, frame_rate,
    reward_x, reward_y, reward_zone,
    table_radius, table_center_x, table_center_y,
    x_start, y_start, start_zone,
    total_distance,
    calculate_relative_distance,
    n_permutations=1000,
    seed=42
):
    """Métriques observées + test de permutation sur les t premières secondes après le départ.
    Les valeurs observées sont renvoyées pour pouvoir les mettre dans le Summary_table.
    NB : la distance utilisée pour crossings/m est celle de la FENÊTRE (comme dans la version d'Aurélie)."""
    individual_x_trim = individual_x[start_frame:int(start_frame + (t * frame_rate))]
    individual_y_trim = individual_y[start_frame:int(start_frame + (t * frame_rate))]
    individual_x_filt = individual_x_trim[~np.isnan(individual_x_trim)]
    individual_y_filt = individual_y_trim[~np.isnan(individual_y_trim)]

    window_duration = len(individual_x_trim) / frame_rate           # durée réelle de la fenêtre (s)
    window_distance, _ = calculate_distance_run(individual_x_trim, individual_y_trim)
    window_distance = max(window_distance, 1)                        # évite division par 0

    # ========== Observed metrics ==========
    enter_reward_zone = 0
    consecutive_count = 0
    first_entry_latency = window_duration
    first_entry_frame_on_filtered = None
    time_spent_in_zone = 0
    crossings_per_m = 0

    for i, (x, y) in enumerate(zip(individual_x_filt, individual_y_filt)):
        if calculate_relative_distance(x, y, reward_x, reward_y) <= reward_zone:
            time_spent_in_zone += 1
            if first_entry_frame_on_filtered is None:
                first_entry_frame_on_filtered = i
            if consecutive_count == 0:
                enter_reward_zone += 1
                consecutive_count = 1
        else:
            consecutive_count = 0

    if first_entry_frame_on_filtered is not None:
        first_entry_frameX = np.where(individual_x == individual_x_filt[int(first_entry_frame_on_filtered)])
        first_entry_frameY = np.where(individual_y == individual_y_filt[int(first_entry_frame_on_filtered)])
        common = np.intersect1d(first_entry_frameX, first_entry_frameY)
        first_entry_frame = int(common[0])
        first_entry_latency = (first_entry_frame - start_frame) / frame_rate
        crossings_per_m = round(enter_reward_zone / (round(window_distance) / 100), 2)
        time_spent_in_zone = time_spent_in_zone / frame_rate

    # ========== Permuted metrics ==========
    np.random.seed(seed)
    random_time_spent_in_zone = []
    random_first_entry_latency = []
    random_crossings_per_m = []
    random_targets = []

    while len(random_time_spent_in_zone) < n_permutations:
        r = table_radius * np.sqrt(np.random.rand())
        theta = 2 * np.pi * np.random.rand()
        rx = table_center_x + r * np.cos(theta)
        ry = table_center_y + r * np.sin(theta)

        if calculate_relative_distance(rx, ry, reward_x, reward_y) <= reward_zone:
            continue
        if calculate_relative_distance(rx, ry, x_start, y_start) <= start_zone:
            continue

        rand_enter_reward_zone = 0
        consecutive_count = 0
        rand_first_entry_latency = window_duration
        rand_first_entry_frame_on_filtered = None
        rand_time_spent_in_zone = 0
        rand_crossings_per_m = 0

        for i, (x, y) in enumerate(zip(individual_x_filt, individual_y_filt)):
            if calculate_relative_distance(x, y, rx, ry) <= reward_zone:
                rand_time_spent_in_zone += 1
                if rand_first_entry_frame_on_filtered is None:
                    rand_first_entry_frame_on_filtered = i
                if consecutive_count == 0:
                    rand_enter_reward_zone += 1
                    consecutive_count = 1
            else:
                consecutive_count = 0

        if rand_first_entry_frame_on_filtered is not None:
            rand_first_entry_frameX = np.where(individual_x == individual_x_filt[int(rand_first_entry_frame_on_filtered)])
            rand_first_entry_frameY = np.where(individual_y == individual_y_filt[int(rand_first_entry_frame_on_filtered)])
            common = np.intersect1d(rand_first_entry_frameX, rand_first_entry_frameY)
            rand_first_entry_frame = int(common[0])
            rand_first_entry_latency = (rand_first_entry_frame - start_frame) / frame_rate
            rand_crossings_per_m = round(rand_enter_reward_zone / (round(window_distance) / 100), 2)
            rand_time_spent_in_zone = rand_time_spent_in_zone / frame_rate

        random_targets.append([rx, ry])
        random_first_entry_latency.append(rand_first_entry_latency)
        random_crossings_per_m.append(rand_crossings_per_m)
        random_time_spent_in_zone.append(rand_time_spent_in_zone)

    random_time_spent_in_zone = np.array(random_time_spent_in_zone)
    random_crossings_per_m = np.array(random_crossings_per_m)
    random_first_entry_latency = np.array(random_first_entry_latency)

    # ========== P-values ==========
    p_value_time_spent_in_zone = (np.sum(random_time_spent_in_zone >= time_spent_in_zone) + 1) / (n_permutations + 1)
    p_value_crossings_per_m = (np.sum(random_crossings_per_m >= crossings_per_m) + 1) / (n_permutations + 1)
    p_value_first_entry_latency = (np.sum(random_first_entry_latency <= first_entry_latency) + 1) / (n_permutations + 1)

    # ========== Combined performance ==========
    mean_time_spent_in_zone = np.mean(random_time_spent_in_zone)
    std_time_spent_in_zone = np.std(random_time_spent_in_zone)
    mean_crossings_per_m = np.mean(random_crossings_per_m)
    std_crossings_per_m = np.std(random_crossings_per_m)
    mean_first_entry_latency = np.mean(random_first_entry_latency)
    std_first_entry_latency = np.std(random_first_entry_latency)

    z_time_obs = (time_spent_in_zone - mean_time_spent_in_zone) / std_time_spent_in_zone if std_time_spent_in_zone > 0 else 0
    z_crossing_obs = (crossings_per_m - mean_crossings_per_m) / std_crossings_per_m if std_crossings_per_m > 0 else 0
    z_latency_obs = -(first_entry_latency - mean_first_entry_latency) / std_first_entry_latency if std_first_entry_latency > 0 else 0

    observed_perf_3m = z_time_obs + z_crossing_obs + z_latency_obs
    observed_perf_2m = z_time_obs + z_crossing_obs

    z_time_perm = (random_time_spent_in_zone - mean_time_spent_in_zone) / std_time_spent_in_zone if std_time_spent_in_zone > 0 else np.zeros(n_permutations)
    z_crossings_perm = (random_crossings_per_m - mean_crossings_per_m) / std_crossings_per_m if std_crossings_per_m > 0 else np.zeros(n_permutations)
    z_latency_perm = -(random_first_entry_latency - mean_first_entry_latency) / std_first_entry_latency if std_first_entry_latency > 0 else np.zeros(n_permutations)

    random_perf_3m = z_time_perm + z_crossings_perm + z_latency_perm
    random_perf_2m = z_time_perm + z_crossings_perm

    p_value_perf_3m = (np.sum(random_perf_3m >= observed_perf_3m) + 1) / (n_permutations + 1)
    p_value_perf_2m = (np.sum(random_perf_2m >= observed_perf_2m) + 1) / (n_permutations + 1)

    return {
        # valeurs observées sur la fenêtre
        "window_duration_s": window_duration,
        "window_distance_cm": window_distance,
        "time_spent_in_zone_s": time_spent_in_zone,
        "time_spent_in_zone_percent": time_spent_in_zone / window_duration * 100 if window_duration > 0 else np.nan,
        "crossings_in_zone": enter_reward_zone,
        "crossings_per_m": crossings_per_m,
        "first_entry_latency_s": first_entry_latency,
        # moyennes de la distribution aléatoire (utile pour comparer)
        "random_mean_time_spent_in_zone_s": mean_time_spent_in_zone,
        "random_mean_crossings_per_m": mean_crossings_per_m,
        "random_mean_first_entry_latency_s": mean_first_entry_latency,
        # p-values
        "p_value_perf_3m": p_value_perf_3m,
        "p_value_perf_2m": p_value_perf_2m,
        "p_value_time_spent_in_zone": p_value_time_spent_in_zone,
        "p_value_crossings_per_m": p_value_crossings_per_m,
        "p_value_first_entry_latency": p_value_first_entry_latency,
        # z-scores
        "z_time_obs": z_time_obs,
        "z_crossing_obs": z_crossing_obs,
        "z_latency_obs": z_latency_obs,
        "performance_3m": observed_perf_3m,
        "performance_2m": observed_perf_2m,
    }

def sholl_analysis(x, y, xr, yr, x_mask, y_mask, mask_radius=None, max_radius=100, step=10, buffer=20):
    if mask_radius is not None:
        x, y = apply_circular_mask(x, y, x_mask, y_mask, mask_radius)

    path_coords = np.column_stack((x, y))
    target = np.array([xr, yr])
    dist_to_target = distance.cdist(path_coords, [target]).flatten()

    radii = np.arange(step, max_radius + step, step)
    intersections = np.zeros(len(radii), dtype=int)

    for i, radius in enumerate(radii):
        crossings = np.where((dist_to_target[:-1] < radius) & (dist_to_target[1:] >= radius))[0]
        intersections[i] = len(crossings)

    radii_cm = radii / pixel_to_cm
    norm_intersections = intersections / (2 * np.pi * radii_cm)

    return norm_intersections, radii, intersections

def detect_immobility(x, y, radius_px, min_stay_frames):
    """Masque booléen des frames immobiles : la souris reste dans un cercle de rayon radius_px
    (centré sur sa position au début de la fenêtre) pendant >= min_stay_frames frames consécutives.
    Même logique que DLC_5_create_table_final_filtered (la fenêtre est testée jusqu'à la toute dernière frame)."""
    mask = np.zeros(len(x), dtype=bool)
    i = 0
    while i <= len(x) - min_stay_frames:
        if np.isnan(x[i]) or np.isnan(y[i]):
            i += 1
            continue
        cx, cy = x[i], y[i]
        d = np.sqrt((x[i:i + min_stay_frames] - cx) ** 2 + (y[i:i + min_stay_frames] - cy) ** 2)
        if np.all(np.where(np.isnan(d), False, d <= radius_px)):
            j = i + min_stay_frames
            while j < len(x) and not np.isnan(x[j]) and np.sqrt((x[j] - cx) ** 2 + (y[j] - cy) ** 2) <= radius_px:
                j += 1
            mask[i:j] = True
            i = j
        else:
            i += 1
    return mask

Summary_Table_Opto       = pd.DataFrame()
Summary_Table_Chemo_Opto = pd.DataFrame()
Sholl_table_Opto         = pd.DataFrame()
Sholl_table_Chemo_Opto   = pd.DataFrame()

filter_immobility = True         # toujours True dans ce notebook (le notebook sans filtre est un script à part)
suffix = '_immobility'           # ajouté à la fin du nom de tous les tableaux
print('Mode : immobilité retirée (tableaux avec _immobility à la fin)')

##############################################################################
# Loop through all .h5 files in the folder
##############################################################################

for root, dirs, files in os.walk(folder_path):
    h5_files = [f for f in files if f.endswith('.h5')]

    for h5_file in h5_files:
        filename = os.path.join(root, h5_file)
        filename = os.path.normpath(filename)

        ########################
        # Extract metadata from the file path
        ########################
        # Cheeseboard / <experiment> / <groupe> / <souris> / <age> / <session_type> / <date> / fichier.h5
        #   experiment = Opto ou Opto_Chemo
        #   groupe     = WT / APPPS1 (Opto)   ou   hM4D / mCherry (Opto_Chemo)   -> colonne 'genotype'
        parents = Path(filename).parents
        if len(parents) < 7:
            n_ignored_other_layout += 1
            continue
        task_type     = os.path.basename(parents[6])
        experiment    = os.path.basename(parents[5])
        genotype      = os.path.basename(parents[4])
        mice          = os.path.basename(parents[3])
        age           = os.path.basename(parents[2])
        session_type  = os.path.basename(parents[1])
        sesstion_time = os.path.basename(parents[0])

        if "Cheeseboard" not in task_type or experiment not in experiment_tables:
            n_ignored_other_layout += 1
            continue

        print(f'Analyzing {experiment} | {genotype} | mouse {mice}, {age}, {session_type} session')

        # Infos de la souris (fichier _MouseID_ImplantColorCode.csv) : sexe, implant (+ génotype pour Opto_Chemo)
        # 'virus' = nom du dossier (Opto_Chemo uniquement)
        group_folder = genotype            # nom du dossier de groupe (WT / APPPS1 / hM4D / mCherry)
        info = mouse_info.get(str(mice).strip())
        sex, implant = None, None                   # None (et non NaN) : garde des colonnes texte
        virus = group_folder if experiment == 'Opto_Chemo' else None   # virus = nom du dossier (hM4D / mCherry)
        if info is None:
            if mouse_info:
                mice_not_in_info.add(mice)
            if experiment == 'Opto_Chemo':
                genotype = None                     # le génotype vient du fichier pour Opto_Chemo
        else:
            sex     = None if pd.isna(info['Sex'])     else info['Sex']
            implant = None if pd.isna(info['Implant']) else info['Implant']

            # contrôle de cohérence avec le nom des dossiers
            if experiment == 'Opto' and str(info['Genotype']) != group_folder:
                info_mismatches.append((mice, f'dossier genotype = {group_folder} / fichier = {info["Genotype"]}'))
            if experiment == 'Opto_Chemo' and not str(info['Virus']).startswith(group_folder):
                info_mismatches.append((mice, f'dossier virus = {group_folder} / fichier = {info["Virus"]}'))

            # Opto_Chemo : toutes les souris sont APPPS1 -> 'genotype' vient du fichier, le groupe d'intérêt est 'virus'
            if experiment == 'Opto_Chemo':
                genotype = info['Genotype']

        ########################
        # Update day/trial counters
        ########################
        if mice == previousmice and experiment == previous_experiment:
            if session_type == previous_session_type:
                if sesstion_time == previous_session_time:
                    trial += 1
                else:
                    day += 1
                    trial = 1
            else:
                day = 1
                trial = 1
        else:
            day = 1
            trial = 1

        ########################
        # Define parameters
        ########################
        with open("//10.69.168.1/crnldata/forgetting/Carla/Cheeseboard/Reward_position.txt", "r") as file:
            text = file.read()
        numbers = re.findall(r"[-+]?\d*\.\d+|\d+", text)
        reward_x, reward_y = map(float, numbers)

        pixel_to_cm = 5.05  # initial placeholder, overwritten below
        table_center_x, table_center_y = 920, 637
        table_radius = 606 / 2
        pixel_to_cm = table_radius / 60  # table is 60 cm radius in real life

        reward_zone     = 8 * pixel_to_cm if ('Training' in session_type or 'Post' in session_type) else 20 * pixel_to_cm
        start_zone      = 30 * pixel_to_cm  # 30 cm start zone in pixels
        min_stay_at_reward_s = 2             # seconds
        table_margin    = 5 * pixel_to_cm   # 5 cm margin to account for head pitch
        

        ########################
        # Load HDF5 file
        ########################
        df = pd.read_hdf(filename)
        directory = os.path.dirname(filename)
        timestamps_path = Path(directory, 'timeStamps.csv')

        # FIX: always initialise start_time / end_time / duration_trial
        start_time = np.nan
        end_time   = np.nan

        if timestamps_path.exists():
            timestamps = pd.read_csv(timestamps_path)
            frame_rate = round(1 / (np.mean(np.diff(timestamps.iloc[:, 1])) / 1000))
        else:
            frame_rate = 16  # /!\ change if needed

        min_stay_at_reward = min_stay_at_reward_s * frame_rate

        ########################
        # Clean the signal
        ########################
        # Qualité de détection : % de frames où la souris est vue avec une vraisemblance > 0.5
        pct_frames_detected = round((df.iloc[:, 2] > 0.5).mean() * 100, 2)
        n_frames_total = len(df)
        mean_likelihood = round(df.iloc[:, 2].mean(), 4)

        df.iloc[:, 0] = df.apply(lambda row: row.iloc[0] if row.iloc[2] > 0.5 else np.nan, axis=1)
        df.iloc[:, 1] = df.apply(lambda row: row.iloc[1] if row.iloc[2] > 0.5 else np.nan, axis=1)

        individual_xO = np.array(df.iloc[:, 0].values, dtype=float)
        individual_yO = np.array(df.iloc[:, 1].values, dtype=float)

        # Keep only points on the cheeseboard
        for i in range(len(individual_xO)):
            if np.isnan(individual_xO[i]) or np.isnan(individual_yO[i]):
                continue
            if calculate_relative_distance(individual_xO[i], individual_yO[i], table_center_x, table_center_y) >= table_radius:
                individual_xO[i] = np.nan
                individual_yO[i] = np.nan

        individual_xOO = remove_short_sequences(individual_xO, max_len=3)
        individual_yOO = remove_short_sequences(individual_yO, max_len=3)

        sequences_x = find_long_non_nan_sequences(individual_xOO)
        sequences_y = find_long_non_nan_sequences(individual_yOO)

        if len(sequences_x) == 0 or len(sequences_y) == 0:
            print(f'  /!\\ SKIPPED : no valid tracking sequence found in {filename}')
            skipped_files.append({
                'experiment': experiment, 'mice': mice, 'age': age, 'genotype': genotype, 'session_type': session_type,
                'session_time': sesstion_time, 'file': filename,
                'reason': 'no valid tracking sequence (souris non détectée)',
                'pct_frames_detected': pct_frames_detected, 'n_frames_total': n_frames_total,
                'mean_likelihood': mean_likelihood
            })
            continue

        x_start = sequences_x[0][0]
        y_start = sequences_y[0][0]

        start_frame = np.where(individual_xOO == x_start)[0][0].item()
        individual_xOO[:start_frame] = np.nan
        individual_yOO[:start_frame] = np.nan

        individual_x1, individual_y1 = replace_high_speed_points_with_nan(individual_xOO, individual_yOO, speed_threshold=10)

        last_frame = start_frame  # fallback
        for i in range(len(individual_x1) - 1, 0, -1):
            if not np.isnan(individual_x1[i]) and not np.isnan(individual_x1[i - 1]):
                last_frame = i
                break

        if last_frame <= start_frame + 1:
            print(f'  /!\\ SKIPPED : empty trajectory after cleaning in {filename}')
            skipped_files.append({
                'experiment': experiment, 'mice': mice, 'age': age, 'genotype': genotype, 'session_type': session_type,
                'session_time': sesstion_time, 'file': filename,
                'reason': 'empty trajectory after cleaning (trajet vide après nettoyage)',
                'pct_frames_detected': pct_frames_detected, 'n_frames_total': n_frames_total,
                'mean_likelihood': mean_likelihood
            })
            continue

        individual_x2, individual_y2 = interpolate_2d_path(individual_x1[start_frame:last_frame], individual_y1[start_frame:last_frame], kind='nearest')
        individual_x3, individual_y3 = limit_speed(individual_x2, individual_y2, max_speed=20)

        individual_x = np.concatenate((individual_x1[:start_frame], individual_x3))
        individual_y = np.concatenate((individual_y1[:start_frame], individual_y3))

        ########################
        # Immobilité : retirée de la trajectoire (uniquement pour les tableaux _immobility)
        ########################
        individual_x_full = individual_x.copy()      # trajectoire complète (sert à détecter la récompense mangée)
        individual_y_full = individual_y.copy()
        immobility_mask = np.zeros(len(individual_x), dtype=bool)
        if filter_immobility:
            immobility_mask = detect_immobility(individual_x, individual_y,
                                                immobility_radius_cm * pixel_to_cm,
                                                int(round(immobility_min_s * frame_rate)))
            individual_x[immobility_mask] = np.nan
            individual_y[immobility_mask] = np.nan

        ########################
        # Compute common metrics
        ########################
        start_quadrant = get_quadrant(x_start, y_start, table_center_x, table_center_y)
        reward_quadrant = get_quadrant(reward_x, reward_y, table_center_x, table_center_y)

        # FIX: compute start_time / end_time / duration_trial here, after start_frame is known
        if timestamps_path.exists():
            start_time = timestamps.iloc[start_frame, 1].item() / 1000
            end_time   = timestamps.iloc[-1, 1].item() / 1000
            duration_trial = end_time - start_time
        else:
            duration_trial = (last_frame - start_frame) / frame_rate

        total_distance, distances = calculate_distance_run(
            individual_x[start_frame:last_frame],
            individual_y[start_frame:last_frame]
        )
        speed = np.nanmean(distances) / pixel_to_cm * frame_rate

        individual_x_filt = individual_x[~np.isnan(individual_x)]
        individual_y_filt = individual_y[~np.isnan(individual_y)]

        # Time spent at table border (8 cm band)
        border = 8 * pixel_to_cm
        border_inner = table_radius - border
        border_outer = table_radius + table_margin
        time_spent_at_border = 0
        for x, y in zip(individual_x_filt, individual_y_filt):
            dist = calculate_relative_distance(x, y, table_center_x, table_center_y)
            if border_inner <= dist <= border_outer:
                time_spent_at_border += 1
        time_spent_at_border_sec = time_spent_at_border / frame_rate

        # Time spent in each quadrant (all session types)
        time_spent_south = time_spent_west = time_spent_north = time_spent_east = 0
        for x, y in zip(individual_x_filt, individual_y_filt):
            dist = calculate_relative_distance(x, y, table_center_x, table_center_y)
            if dist <= table_radius:
                q = get_quadrant(x, y, table_center_x, table_center_y)
                if q == "South": time_spent_south += 1
                elif q == "West":  time_spent_west  += 1
                elif q == "North": time_spent_north += 1
                else:              time_spent_east  += 1
        time_spent_south_sec = time_spent_south / frame_rate
        time_spent_west_sec  = time_spent_west  / frame_rate
        time_spent_north_sec = time_spent_north / frame_rate
        time_spent_east_sec  = time_spent_east  / frame_rate

        ########################
        # Reward-zone metrics (ALL session types, comme la version d'Aurélie)
        ########################
        enter_reward_zone = 0
        consecutive_count = 0
        first_entry_latency = (last_frame - start_frame) / frame_rate
        first_entry_frame_on_filtered = None
        time_spent_in_zone = 0
        crossings_per_m = 0

        for i, (x, y) in enumerate(zip(individual_x_filt, individual_y_filt)):
            if calculate_relative_distance(x, y, reward_x, reward_y) <= reward_zone:
                time_spent_in_zone += 1
                if first_entry_frame_on_filtered is None:
                    first_entry_frame_on_filtered = i
                if consecutive_count == 0:
                    enter_reward_zone += 1
                    consecutive_count = 1
            else:
                consecutive_count = 0

        if first_entry_frame_on_filtered is not None:
            first_entry_frameX = np.where(individual_x == individual_x_filt[int(first_entry_frame_on_filtered)])
            first_entry_frameY = np.where(individual_y == individual_y_filt[int(first_entry_frame_on_filtered)])
            common = np.intersect1d(first_entry_frameX, first_entry_frameY)
            first_entry_frame = int(common[0])
            first_entry_latency = (first_entry_frame - start_frame) / frame_rate
            crossings_per_m = round(enter_reward_zone / (round(total_distance) / 100), 2) if round(total_distance) > 0 else np.nan
            time_spent_in_zone = time_spent_in_zone / frame_rate

        ########################
        # Training-specific metrics
        ########################
        latency_to_reward  = np.nan
        distance_to_reward = np.nan
        speed_to_reward    = np.nan
        found_reward_time  = np.nan
        found_reward_frame = None

        if filter_immobility:
            # La récompense est 'mangée' en restant immobile (>= 2 s dans la zone) : sa détection se fait donc
            # sur la trajectoire COMPLÈTE, sinon ces frames seraient retirées et la récompense jamais trouvée.
            individual_x_masked, individual_y_masked = individual_x, individual_y
            individual_x_filt_masked, individual_y_filt_masked = individual_x_filt, individual_y_filt
            individual_x, individual_y = individual_x_full, individual_y_full
            individual_x_filt = individual_x[~np.isnan(individual_x)]
            individual_y_filt = individual_y[~np.isnan(individual_y)]

        if 'Training' in session_type or 'Post' in session_type:
            X = df.iloc[:, 0]

            eat_reward_frame_on_filtered = np.nan
            enter_reward_zone_tr = 0
            consecutive_count = 0
            timespent = 0
            eat_reward = False

            for i, (x, y) in enumerate(zip(individual_x_filt, individual_y_filt)):
                if calculate_relative_distance(x, y, reward_x, reward_y) <= reward_zone:
                    timespent += 1
                    if consecutive_count == 0:
                        enter_reward_zone_tr += 1
                        consecutive_count = 1
                else:
                    consecutive_count = 0
                    timespent = 0
                if timespent > min_stay_at_reward:
                    eat_reward_frame_on_filtered = i - min_stay_at_reward
                    eat_reward = True
                    break

            # Fallback if reward not detected long enough
            if not eat_reward and len(X) / frame_rate < 250:
                print('Reward eaten, but not detected long enough (detection problem)')
                All_timespent = pd.DataFrame(columns=['timespent_in_zone', 'frame_i'])
                enter_reward_zone_tr = 0
                consecutive_count = 0
                timespent = 0
                eat_reward = True
                for i, (x, y) in enumerate(zip(individual_x_filt, individual_y_filt)):
                    if calculate_relative_distance(x, y, reward_x, reward_y) <= reward_zone:
                        timespent += 1
                        if consecutive_count == 0:
                            enter_reward_zone_tr += 1
                            consecutive_count = 1
                        All_timespent.loc[enter_reward_zone_tr, 'timespent_in_zone'] = timespent
                        All_timespent.loc[enter_reward_zone_tr, 'frame_i'] = i - timespent
                    else:
                        consecutive_count = 0
                        timespent = 0
                if not All_timespent.empty:
                    eat_reward_frame_on_filtered = int(All_timespent.loc[All_timespent['timespent_in_zone'].idxmax(), 'frame_i'])

            if eat_reward and not np.isnan(eat_reward_frame_on_filtered):
                found_reward_frameX = np.where(individual_x == individual_x_filt[int(eat_reward_frame_on_filtered)])
                found_reward_frameY = np.where(individual_y == individual_y_filt[int(eat_reward_frame_on_filtered)])
                common = np.intersect1d(found_reward_frameX, found_reward_frameY)
                found_reward_frame = int(common[0])
                if timestamps_path.exists():
                    found_reward_time = timestamps.iloc[found_reward_frame, 1].item() / 1000
                    latency_to_reward = found_reward_time - start_time
                else:
                    latency_to_reward = (found_reward_frame - start_frame) / frame_rate
                distance_to_reward, distances_to_reward = calculate_distance_run(
                    individual_x[start_frame:found_reward_frame],
                    individual_y[start_frame:found_reward_frame]
                )
                speed_to_reward = np.nanmean(distances_to_reward) / pixel_to_cm * frame_rate

        if filter_immobility:
            # retour à la trajectoire SANS immobilité pour tout le reste
            individual_x, individual_y = individual_x_masked, individual_y_masked
            individual_x_filt, individual_y_filt = individual_x_filt_masked, individual_y_filt_masked
            # distance / vitesse jusqu'à la récompense : calculées hors immobilité (la latence reste en temps réel)
            if found_reward_frame is not None:
                distance_to_reward, distances_to_reward = calculate_distance_run(
                    individual_x[start_frame:found_reward_frame],
                    individual_y[start_frame:found_reward_frame]
                )
                speed_to_reward = np.nanmean(distances_to_reward) / pixel_to_cm * frame_rate

        ########################
        # Permutation test + Sholl (Test / Probe / Habituation), fenêtres 2 min ET 3 min
        ########################
        is_probe_like = any(k in session_type for k in ('Test', 'Probe', 'Habituation'))
        windows = {'2min': 60 * 2, '3min': 60 * 3}
        perm_results = {}
        sholl_results = {}

        if is_probe_like:
            for label, t in windows.items():
                perm_results[label] = compute_performance_metrics(
                    individual_x=individual_x,
                    individual_y=individual_y,
                    start_frame=start_frame,
                    t=t,
                    frame_rate=frame_rate,
                    reward_x=reward_x,
                    reward_y=reward_y,
                    reward_zone=reward_zone,
                    table_radius=table_radius,
                    table_center_x=table_center_x,
                    table_center_y=table_center_y,
                    x_start=x_start,
                    y_start=y_start,
                    start_zone=start_zone,
                    total_distance=total_distance,
                    calculate_relative_distance=calculate_relative_distance,
                    n_permutations=1000,
                    seed=42
                )
                sholl_results[label] = sholl_analysis(
                    individual_x[start_frame:int(start_frame + (t * frame_rate))],
                    individual_y[start_frame:int(start_frame + (t * frame_rate))],
                    reward_x, reward_y,
                    table_center_x, table_center_y,
                    mask_radius=table_radius,
                    max_radius=table_radius * 2,
                    step=10
                )

        ########################
        # Fill Summary table
        ########################
        Summary_table.loc[counter, 'mice']         = mice
        Summary_table.loc[counter, 'age']          = age
        Summary_table.loc[counter, 'genotype']     = genotype
        Summary_table.loc[counter, 'sex']          = sex
        Summary_table.loc[counter, 'implant']      = implant
        Summary_table.loc[counter, 'virus']        = virus
        Summary_table.loc[counter, 'task_type']    = task_type
        Summary_table.loc[counter, 'experiment']   = experiment
        Summary_table.loc[counter, 'session_type'] = session_type
        Summary_table.loc[counter, 'session']      = int(day)
        Summary_table.loc[counter, 'session_time'] = sesstion_time
        Summary_table.loc[counter, 'trial']        = int(trial)
        Summary_table.loc[counter, 'file']         = filename      # pour retrouver la vidéo

        # Qualité du tracking
        Summary_table.loc[counter, 'frame_rate']          = frame_rate
        Summary_table.loc[counter, 'n_frames_total']      = n_frames_total
        Summary_table.loc[counter, 'pct_frames_detected'] = pct_frames_detected
        Summary_table.loc[counter, 'mean_likelihood']     = mean_likelihood

        Summary_table.loc[counter, 'start_quadrant_loc'] = start_quadrant
        Summary_table.loc[counter, 'start_time']         = start_time
        Summary_table.loc[counter, 'end_time']           = end_time

        Summary_table.loc[counter, 'reward_location_pix']    = str([reward_x, reward_y])
        Summary_table.loc[counter, 'reward_zone_radius_cm']  = reward_zone / pixel_to_cm
        Summary_table.loc[counter, 'reward_quadrant_loc']    = reward_quadrant

        # Locomotion
        Summary_table.loc[counter, 'duration_trial_s']             = round(duration_trial, 2)
        Summary_table.loc[counter, 'total_distance_cm']            = round(total_distance, 2)
        Summary_table.loc[counter, 'average_speed_cm_s']           = round(speed, 2)                           # moyenne des vitesses image par image
        Summary_table.loc[counter, 'average_speed_total_cm_s']     = round(total_distance / duration_trial, 2) # distance totale / durée (version Aurélie)
        Summary_table.loc[counter, 'time_spent_at_border_s']       = round(time_spent_at_border_sec, 2)
        Summary_table.loc[counter, 'time_spent_at_border_percent'] = round(time_spent_at_border_sec / duration_trial * 100, 2)

        # Immobilité (tableaux _immobility uniquement)
        if filter_immobility:
            immobility_time_s = immobility_mask.sum() / frame_rate
            n_bouts = int(np.sum(np.diff(np.concatenate(([0], immobility_mask.astype(int)))) == 1))
            Summary_table.loc[counter, 'immobility_time_s']       = round(immobility_time_s, 2)
            Summary_table.loc[counter, 'immobility_percent']      = round(immobility_time_s / duration_trial * 100, 2)
            Summary_table.loc[counter, 'n_immobility_bouts']      = n_bouts
            Summary_table.loc[counter, 'immobility_radius_cm']    = immobility_radius_cm
            Summary_table.loc[counter, 'immobility_min_s']        = immobility_min_s

        # Quadrants
        Summary_table.loc[counter, 'time_in_S_percent'] = round(time_spent_south_sec / duration_trial * 100, 2)
        Summary_table.loc[counter, 'time_in_W_percent'] = round(time_spent_west_sec  / duration_trial * 100, 2)
        Summary_table.loc[counter, 'time_in_N_percent'] = round(time_spent_north_sec / duration_trial * 100, 2)
        Summary_table.loc[counter, 'time_in_E_percent'] = round(time_spent_east_sec  / duration_trial * 100, 2)

        # Zone de récompense, tout l'essai (toutes les sessions : Training, Test, Habituation...)
        Summary_table.loc[counter, 'time_spent_in_zone_s']       = round(time_spent_in_zone, 2)
        Summary_table.loc[counter, 'time_spent_in_zone_percent'] = round(time_spent_in_zone / duration_trial * 100, 2)
        Summary_table.loc[counter, 'crossings_in_zone']          = enter_reward_zone
        Summary_table.loc[counter, 'crossings_in_zone_per_m']    = round(crossings_per_m, 2)
        Summary_table.loc[counter, 'first_entry_in_zone_s']      = round(first_entry_latency, 2)

        if 'Training' in session_type or 'Post' in session_type:
            Summary_table.loc[counter, 'found_reward_time']             = found_reward_time
            Summary_table.loc[counter, 'average_speed_to_reward_cm_s']  = round(speed_to_reward, 2) if not np.isnan(speed_to_reward) else np.nan
            Summary_table.loc[counter, 'latency_to_reward_s']           = round(latency_to_reward, 2) if not np.isnan(latency_to_reward) else np.nan
            Summary_table.loc[counter, 'distance_to_reward_cm']         = round(distance_to_reward, 2) if not np.isnan(distance_to_reward) else np.nan

        if is_probe_like:
            for label, res in perm_results.items():
                s = f'_{label}'
                # Valeurs observées sur la fenêtre
                Summary_table.loc[counter, 'window_duration_s' + s]            = round(res['window_duration_s'], 2)
                Summary_table.loc[counter, 'window_distance_cm' + s]           = round(res['window_distance_cm'], 2)
                Summary_table.loc[counter, 'time_spent_in_zone_s' + s]         = round(res['time_spent_in_zone_s'], 2)
                Summary_table.loc[counter, 'time_spent_in_zone_percent' + s]   = round(res['time_spent_in_zone_percent'], 2)
                Summary_table.loc[counter, 'crossings_in_zone' + s]            = res['crossings_in_zone']
                Summary_table.loc[counter, 'crossings_in_zone_per_m' + s]      = round(res['crossings_per_m'], 2)
                Summary_table.loc[counter, 'first_entry_in_zone_s' + s]        = round(res['first_entry_latency_s'], 2)
                # Distribution aléatoire (moyennes)
                Summary_table.loc[counter, 'random_mean_time_in_zone_s' + s]       = round(res['random_mean_time_spent_in_zone_s'], 2)
                Summary_table.loc[counter, 'random_mean_crossings_per_m' + s]      = round(res['random_mean_crossings_per_m'], 2)
                Summary_table.loc[counter, 'random_mean_first_entry_s' + s]        = round(res['random_mean_first_entry_latency_s'], 2)
                # z-scores et performances
                Summary_table.loc[counter, 'z_time_obs' + s]      = round(res['z_time_obs'], 4)
                Summary_table.loc[counter, 'z_crossing_obs' + s]  = round(res['z_crossing_obs'], 4)
                Summary_table.loc[counter, 'z_latency_obs' + s]   = round(res['z_latency_obs'], 4)
                Summary_table.loc[counter, 'performance_3m' + s]  = round(res['performance_3m'], 4)
                Summary_table.loc[counter, 'performance_2m' + s]  = round(res['performance_2m'], 4)
                # p-values
                Summary_table.loc[counter, 'p_value_perf3m' + s]                  = round(res['p_value_perf_3m'], 4)
                Summary_table.loc[counter, 'p_value_perf2m' + s]                  = round(res['p_value_perf_2m'], 4)
                Summary_table.loc[counter, 'p_value_time_in_zone' + s]            = round(res['p_value_time_spent_in_zone'], 4)
                Summary_table.loc[counter, 'p_value_crossings_in_zone_per_m' + s] = round(res['p_value_crossings_per_m'], 4)
                Summary_table.loc[counter, 'p_value_first_entry_latency' + s]     = round(res['p_value_first_entry_latency'], 4)

            ###################################################
            # Sholl Analysis (normalisé ET brut, 2 min et 3 min)
            ###################################################
            Sholl_table.loc[counterProbe, 'task_type']    = task_type
            Sholl_table.loc[counterProbe, 'experiment']   = experiment
            Sholl_table.loc[counterProbe, 'session_type'] = session_type
            Sholl_table.loc[counterProbe, 'mice']         = mice
            Sholl_table.loc[counterProbe, 'age']          = age
            Sholl_table.loc[counterProbe, 'genotype']     = genotype
            Sholl_table.loc[counterProbe, 'sex']          = sex
            Sholl_table.loc[counterProbe, 'implant']      = implant
            Sholl_table.loc[counterProbe, 'virus']        = virus
            Sholl_table.loc[counterProbe, 'session']      = int(day)
            Sholl_table.loc[counterProbe, 'session_time'] = sesstion_time
            Sholl_table.loc[counterProbe, 'trial']        = int(trial)

            for label, (norm_inter, radii, raw_inter) in sholl_results.items():
                for i in range(len(radii)):
                    Sholl_table.loc[counterProbe, f'norm_intersec_{int(radii[i])}rad_{label}'] = norm_inter[i]
                    Sholl_table.loc[counterProbe, f'intersec_{int(radii[i])}rad_{label}']      = raw_inter[i]
            counterProbe += 1

        # Update trackers
        previousmice          = mice
        previous_session_time = sesstion_time
        previous_session_type = session_type
        previous_experiment   = experiment
        counter += 1

##############################################################################
# Save output tables : un tableau par expérience
#   Opto       -> Summary_Table_Opto       / Sholl_analysis_table_Opto
#   Opto_Chemo -> Summary_Table_Chemo_Opto / Sholl_analysis_table_Chemo_Opto
#   (noms avec '_immobility' à la fin)
##############################################################################
for exp_folder, exp_name in experiment_tables.items():
    if 'experiment' in Summary_table.columns:
        summ = Summary_table[Summary_table['experiment'] == exp_folder].reset_index(drop=True)
    else:
        summ = pd.DataFrame()
    if 'experiment' in Sholl_table.columns:
        sholl = Sholl_table[Sholl_table['experiment'] == exp_folder].reset_index(drop=True)
    else:
        sholl = pd.DataFrame()

    # pas de colonne 'virus' pour Opto (vide)
    for _df in (summ, sholl):
        if 'virus' in _df.columns and _df['virus'].isna().all():
            _df.drop(columns='virus', inplace=True)

    globals()[f'Summary_Table_{exp_name}{suffix}'] = summ
    globals()[f'Sholl_table_{exp_name}{suffix}']   = sholl

    if len(summ) == 0:
        print(f'/!\\ Aucune vidéo analysée pour {exp_folder} : pas de Summary_Table_{exp_name}{suffix}')
        continue

    filenameOut = f'{folder_path}/Summary_Table_{exp_name}{suffix}.xlsx'
    with pd.ExcelWriter(filenameOut) as writer:
        summ.to_excel(writer, index=False)
    print(f'Summary table saved : {filenameOut}  ({len(summ)} rows)')

    if len(sholl) > 0:
        filenameOut = f'{folder_path}/Sholl_analysis_table_{exp_name}{suffix}.xlsx'
        with pd.ExcelWriter(filenameOut) as writer:
            sholl.to_excel(writer, index=False)
        print(f'Sholl table saved   : {filenameOut}  ({len(sholl)} rows)')

if mice_not_in_info:
    print(f'\n/!\\ Souris absentes du fichier _MouseID_ImplantColorCode.csv (sex / implant vides) : {sorted(mice_not_in_info)}')
if info_mismatches:
    print('\n/!\\ Incohérences entre les dossiers et le fichier souris :')
    for m, msg in sorted(set(info_mismatches)):
        print(f'   {m} : {msg}')

if n_ignored_other_layout > 0:
    print(f'\n({n_ignored_other_layout} fichier(s) .h5 ignoré(s) : hors Cheeseboard/Opto et Cheeseboard/Opto_Chemo)')

##############################################################################
# Vidéos NON analysées (souris non détectée, trajet vide...)
##############################################################################
Skipped_table = pd.DataFrame(skipped_files)
if len(Skipped_table) > 0:
    print(f'\n/!\\ {len(Skipped_table)} vidéo(s) NON analysée(s) :')
    display(Skipped_table[['experiment', 'mice', 'age', 'genotype', 'session_type', 'session_time', 'reason', 'pct_frames_detected']])
    filenameOut = f'{folder_path}/Skipped_videos.xlsx'
    Skipped_table.to_excel(filenameOut, index=False)
    print(f'Skipped table saved : {filenameOut}')
else:
    print('\nToutes les vidéos ont été analysées.')


Fichier souris chargé : 48 souris (C:/Users/AudreyHay/Documents/Carla/Visual Code/HayLabAnalysis/python/_MouseID_ImplantColorCode.csv)
Mode : immobilité retirée (tableaux avec _immobility à la fin)
Analyzing Opto | APPPS1 | mouse AHAD01.37, 10months, Post session
Analyzing Opto | APPPS1 | mouse AHAD01.37, 10months, Post session


C:\Users\AudreyHay\AppData\Local\Temp\ipykernel_13768\1739641.py:775: RuntimeWarning: Mean of empty slice
  speed_to_reward = np.nanmean(distances_to_reward) / pixel_to_cm * frame_rate
C:\Users\AudreyHay\AppData\Local\Temp\ipykernel_13768\1739641.py:787: RuntimeWarning: Mean of empty slice
  speed_to_reward = np.nanmean(distances_to_reward) / pixel_to_cm * frame_rate


Analyzing Opto | APPPS1 | mouse AHAD01.37, 10months, Post session
  /!\ SKIPPED : no valid tracking sequence found in \\10.69.168.1\crnldata\forgetting\Carla\Cheeseboard\Opto\APPPS1\AHAD01.37\10months\Post\06_11_2025\AHAD01.37-AfterTest9-3DLC_Resnet50_CheeseboardFeb6shuffle1_snapshot_010_old.h5
Analyzing Opto | APPPS1 | mouse AHAD01.37, 10months, Test session
Analyzing Opto | APPPS1 | mouse AHAD01.37, 11months, Post session
Analyzing Opto | APPPS1 | mouse AHAD01.37, 11months, Post session
Analyzing Opto | APPPS1 | mouse AHAD01.37, 11months, Post session
  /!\ SKIPPED : no valid tracking sequence found in \\10.69.168.1\crnldata\forgetting\Carla\Cheeseboard\Opto\APPPS1\AHAD01.37\11months\Post\09_12_2025\AHAD01.37-AfterTest10-3DLC_Resnet50_CheeseboardFeb6shuffle1_snapshot_010_old.h5
Analyzing Opto | APPPS1 | mouse AHAD01.37, 11months, Test session
Analyzing Opto | APPPS1 | mouse AHAD01.37, 12months, Post session
Analyzing Opto | APPPS1 | mouse AHAD01.37, 12months, Post session
Analyzing O

C:\Users\AudreyHay\AppData\Local\Temp\ipykernel_13768\1739641.py:637: RuntimeWarning: Mean of empty slice
  speed = np.nanmean(distances) / pixel_to_cm * frame_rate
C:\Users\AudreyHay\AppData\Local\Temp\ipykernel_13768\1739641.py:775: RuntimeWarning: Mean of empty slice
  speed_to_reward = np.nanmean(distances_to_reward) / pixel_to_cm * frame_rate
C:\Users\AudreyHay\AppData\Local\Temp\ipykernel_13768\1739641.py:787: RuntimeWarning: Mean of empty slice
  speed_to_reward = np.nanmean(distances_to_reward) / pixel_to_cm * frame_rate


Analyzing Opto | APPPS1 | mouse AHAD01.37, 3months, Training session
  /!\ SKIPPED : no valid tracking sequence found in \\10.69.168.1\crnldata\forgetting\Carla\Cheeseboard\Opto\APPPS1\AHAD01.37\3months\Training\26_03_2025\AHAD01.37-Training J3-2DLC_Resnet50_CheeseboardFeb6shuffle1_snapshot_010_old.h5
Analyzing Opto | APPPS1 | mouse AHAD01.37, 3months, Training session
  /!\ SKIPPED : no valid tracking sequence found in \\10.69.168.1\crnldata\forgetting\Carla\Cheeseboard\Opto\APPPS1\AHAD01.37\3months\Training\26_03_2025\AHAD01.37-Training J3-3DLC_Resnet50_CheeseboardFeb6shuffle1_snapshot_010_old.h5
Analyzing Opto | APPPS1 | mouse AHAD01.37, 3months, Training session
Analyzing Opto | APPPS1 | mouse AHAD01.37, 3months, Training session


C:\Users\AudreyHay\AppData\Local\Temp\ipykernel_13768\1739641.py:775: RuntimeWarning: Mean of empty slice
  speed_to_reward = np.nanmean(distances_to_reward) / pixel_to_cm * frame_rate
C:\Users\AudreyHay\AppData\Local\Temp\ipykernel_13768\1739641.py:787: RuntimeWarning: Mean of empty slice
  speed_to_reward = np.nanmean(distances_to_reward) / pixel_to_cm * frame_rate


... (329 outputs left out)

C:\Users\AudreyHay\AppData\Local\Temp\ipykernel_13768\1739641.py:775: RuntimeWarning: Mean of empty slice
  speed_to_reward = np.nanmean(distances_to_reward) / pixel_to_cm * frame_rate
C:\Users\AudreyHay\AppData\Local\Temp\ipykernel_13768\1739641.py:787: RuntimeWarning: Mean of empty slice
  speed_to_reward = np.nanmean(distances_to_reward) / pixel_to_cm * frame_rate


Analyzing Opto_Chemo | mCherry | mouse AHAD21.157, 5months, Post session
Analyzing Opto_Chemo | mCherry | mouse AHAD21.157, 5months, Test session
Analyzing Opto_Chemo | mCherry | mouse AHAD21.157, 6months, Post session
Analyzing Opto_Chemo | mCherry | mouse AHAD21.157, 6months, Post session
  /!\ SKIPPED : no valid tracking sequence found in \\10.69.168.1\crnldata\forgetting\Carla\Cheeseboard\Opto_Chemo\mCherry\AHAD21.157\6months\Post\09_10_2026\AHAD21.157-AfterTest5-2DLC_Resnet50_CheeseboardFeb6shuffle1_snapshot_010.h5
Analyzing Opto_Chemo | mCherry | mouse AHAD21.157, 6months, Post session
  /!\ SKIPPED : no valid tracking sequence found in \\10.69.168.1\crnldata\forgetting\Carla\Cheeseboard\Opto_Chemo\mCherry\AHAD21.157\6months\Post\09_10_2026\AHAD21.157-AfterTest5-3DLC_Resnet50_CheeseboardFeb6shuffle1_snapshot_010.h5
Analyzing Opto_Chemo | mCherry | mouse AHAD21.157, 6months, Test session
Summary table saved : //10.69.168.1/crnldata/forgetting/Carla/Cheeseboard//Summary_Table_Opto_

,experiment,mice,age,genotype,session_type,session_time,reason,pct_frames_detected
0,Opto,AHAD01.37,10months,APPPS1,Post,06_11_2025,no valid tracking sequence (souris non détectée),88.36
1,Opto,AHAD01.37,11months,APPPS1,Post,09_12_2025,no valid tracking sequence (souris non détectée),95.44
2,Opto,AHAD01.37,2months,APPPS1,Training,24_02_2025,no valid tracking sequence (souris non détectée),81.71
3,Opto,AHAD01.37,2months,APPPS1,Training,27_02_2025,no valid tracking sequence (souris non détectée),73.49
4,Opto,AHAD01.37,3months,APPPS1,Post,27_03_2025,no valid tracking sequence (souris non détectée),97.53
...,...,...,...,...,...,...,...,...
222,Opto_Chemo,AHAD21.155,6months,APPPS1,Post,09_10_2026,no valid tracking sequence (souris non détectée),98.70
223,Opto_Chemo,AHAD21.157,3months,APPPS1,Post,17_07_2026,no valid tracking sequence (souris non détectée),75.63
224,Opto_Chemo,AHAD21.157,3months,APPPS1,Training,15_07_2026,no valid tracking sequence (souris non détectée),69.66
225,Opto_Chemo,AHAD21.157,6months,APPPS1,Post,09_10_2026,no valid tracking sequence (souris non détectée),100.00


Skipped table saved : //10.69.168.1/crnldata/forgetting/Carla/Cheeseboard//Skipped_videos.xlsx


# Tableaux normalisés par souris (3 months = 1)

In [4]:
##############################################################################
# Tableaux normalisés : chaque souris est normalisée par rapport à 3 mois (3 mois = 1)
##############################################################################
import numpy as np
import pandas as pd

ref_age = "3months"      # <- âge de référence (valeur = 1), tel qu'écrit dans le nom du dossier d'âge

# Colonnes d'identification / qualité : gardées telles quelles (non normalisées)
keep_raw = ['mice', 'age', 'genotype', 'task_type', 'experiment', 'session_type', 'session', 'session_time',
            'trial', 'file', 'start_quadrant_loc', 'reward_location_pix', 'reward_quadrant_loc',
            'start_time', 'end_time', 'frame_rate', 'n_frames_total', 'pct_frames_detected',
            'mean_likelihood', 'reward_zone_radius_cm', 'sex', 'implant', 'virus',
            'immobility_radius_cm', 'immobility_min_s']      # paramètres constants : non normalisés


def normalize_to_ref_age(table, ref_age):
    df = table.copy()

    # p-values, z-scores et performances : un ratio n'a pas de sens (valeurs nulles/négatives) -> retirés
    drop_cols = [c for c in df.columns if c.startswith(('p_value', 'z_', 'performance_', 'window_duration'))]
    df = df.drop(columns=drop_cols)

    num_cols = [c for c in df.columns if c not in keep_raw]
    df[num_cols] = df[num_cols].apply(pd.to_numeric, errors='coerce')

    # Valeur de référence par souris et par type de session (moyenne si plusieurs essais à 4 mois)
    ref = df[df['age'] == ref_age].groupby(['mice', 'session_type'])[num_cols].mean()
    ref = ref.where(ref != 0)          # référence = 0 -> division impossible -> NaN

    ref_aligned = ref.reindex(pd.MultiIndex.from_frame(df[['mice', 'session_type']]))
    ref_aligned.index = df.index

    normalized = df.copy()
    normalized[num_cols] = df[num_cols] / ref_aligned

    # Vérifications
    if ref.empty:
        print(f'   /!\\ Aucune ligne avec age == "{ref_age}". Ages présents : {sorted(df["age"].astype(str).unique())}')
    else:
        missing = sorted(set(df['mice']) - set(df.loc[df['age'] == ref_age, 'mice']))
        if missing:
            print(f'   /!\\ Souris sans donnée à {ref_age} (valeurs normalisées = NaN) : {missing}')
    return normalized


suffix = '_immobility'      # tableaux sans immobilité (suffixe à la fin du nom)
for exp_name in ['Opto', 'Chemo_Opto']:

    # Si le notebook a été relancé, on recharge le tableau sauvegardé
    var_name = f'Summary_Table_{exp_name}{suffix}'
    if var_name in globals() and len(globals()[var_name]) > 0:
        table = globals()[var_name]
    else:
        path_in = f'{folder_path}/Summary_Table_{exp_name}{suffix}.xlsx'
        if os.path.exists(path_in):
            table = pd.read_excel(path_in)
        else:
            print(f'\n{exp_name}{suffix} : aucun tableau disponible -> ignoré')
            continue

    print(f'\n=== {exp_name}{suffix} ===')
    normalized = normalize_to_ref_age(table, ref_age)
    globals()[f'Summary_Table_{exp_name}_normalized{suffix}'] = normalized

    filenameOut = f'{folder_path}/Summary_Table_{exp_name}_normalized{suffix}.xlsx'
    normalized.to_excel(filenameOut, index=False)
    print(f'Summary_Table_{exp_name}_normalized{suffix} saved : {filenameOut}  ({len(normalized)} rows)')



=== Opto_immobility ===
   /!\ Souris sans donnée à 3months (valeurs normalisées = NaN) : ['AHAD11.102']
Summary_Table_Opto_normalized_immobility saved : //10.69.168.1/crnldata/forgetting/Carla/Cheeseboard//Summary_Table_Opto_normalized_immobility.xlsx  (3447 rows)

=== Chemo_Opto_immobility ===
Summary_Table_Chemo_Opto_normalized_immobility saved : //10.69.168.1/crnldata/forgetting/Carla/Cheeseboard//Summary_Table_Chemo_Opto_normalized_immobility.xlsx  (433 rows)
